Défi quotidien : Comparaisons de l'attention multiple et du transformateur


Tâche

1. Implémentation de l'attention à une seule tête

 Objectif : Mettre en œuvre le module de base avant de l'étendre à plusieurs têtes.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [6]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.hidden_dim = hidden_dim

# Linear projections for Query, Key, and Value
        self.query_projection = nn.Linear(hidden_dim, hidden_dim)
        self.key_projection = nn.Linear(hidden_dim, hidden_dim)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim)

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        # Linear layers for Q, K, V projections for all heads
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

        # Output linear layer
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, query, key, value):
        batch_size = query.shape[0]

        # 1. Project Q, K, V and split into multiple heads
        # Shape after projection: (batch_size, seq_len, hidden_dim)
        # Reshape to (batch_size, seq_len, num_heads, head_dim)
        # Transpose to (batch_size, num_heads, seq_len, head_dim)
        Q = self.q_proj(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. Calculate attention scores (dot product)
        # scores: (batch_size, num_heads, seq_len_q, seq_len_k)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1))
        attention_scores = attention_scores / (self.head_dim ** 0.5)

        # 3. Apply softmax to get attention weights
        # attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k)
        attention_weights = F.softmax(attention_scores, dim=-1)

        # 4. Multiply by Value to get the weighted sum
        # x: (batch_size, num_heads, seq_len_q, head_dim)
        x = torch.matmul(attention_weights, V)

        # 5. Concatenate heads and apply final linear layer
        # Transpose back to (batch_size, seq_len_q, num_heads, head_dim)
        # Reshape to (batch_size, seq_len_q, hidden_dim)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_dim)

        # Final linear projection
        output = self.out_proj(x)

        return output, attention_weights

# --- Validation with dummy tensors for Multi-Head Attention ---
print("\n--- Validating Multi-Head Attention Module ---")

batch_size = 2
seq_len_q = 5
seq_len_kv = 7
hidden_dim = 64
num_heads = 8 # hidden_dim (64) must be divisible by num_heads (8)

# Create dummy tensors
dummy_query = torch.randn(batch_size, seq_len_q, hidden_dim)
dummy_key = torch.randn(batch_size, seq_len_kv, hidden_dim)
dummy_value = torch.randn(batch_size, seq_len_kv, hidden_dim)

# Instantiate the multi-head attention module
multi_head_attention = MultiHeadAttention(hidden_dim, num_heads)

# Forward pass
output_mha, attention_weights_mha = multi_head_attention(dummy_query, dummy_key, dummy_value)

print(f"Input Query shape: {dummy_query.shape}")
print(f"Input Key shape:   {dummy_key.shape}")
print(f"Input Value shape: {dummy_value.shape}")
print(f"Output MHA shape: {output_mha.shape}") # Expected: (batch_size, seq_len_q, hidden_dim)
print(f"Attention Weights MHA shape: {attention_weights_mha.shape}") # Expected: (batch_size, num_heads, seq_len_q, seq_len_kv)

# Display a sample of attention weights for the first batch, first head, and first query token
print("\nSample Multi-Head Attention Weights (Batch 0, Head 0, Query 0):")
print(attention_weights_mha[0, 0, 0, :])


--- Validating Multi-Head Attention Module ---
Input Query shape: torch.Size([2, 5, 64])
Input Key shape:   torch.Size([2, 7, 64])
Input Value shape: torch.Size([2, 7, 64])
Output MHA shape: torch.Size([2, 5, 64])
Attention Weights MHA shape: torch.Size([2, 8, 5, 7])

Sample Multi-Head Attention Weights (Batch 0, Head 0, Query 0):
tensor([0.0994, 0.1481, 0.1569, 0.1684, 0.1156, 0.1420, 0.1698],
       grad_fn=<SelectBackward0>)


Module d'attention multi-têtes

In [10]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, hidden_dim, ff_dim, dropout_rate=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(hidden_dim, ff_dim)
        self.w_2 = nn.Linear(ff_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        return self.dropout(self.w_2(F.relu(self.w_1(x))))

# --- Validation for PositionwiseFeedForward ---
print("\n--- Validating PositionwiseFeedForward Module ---")
batch_size = 2
seq_len = 5
hidden_dim = 64
ff_dim = 256 # Typically 2 or 4 times hidden_dim

dummy_input_ff = torch.randn(batch_size, seq_len, hidden_dim)
pff_module = PositionwiseFeedForward(hidden_dim, ff_dim)
output_ff = pff_module(dummy_input_ff)

print(f"Input PFF shape: {dummy_input_ff.shape}")
print(f"Output PFF shape: {output_ff.shape}") # Expected: (batch_size, seq_len, hidden_dim)


--- Validating PositionwiseFeedForward Module ---
Input PFF shape: torch.Size([2, 5, 64])
Output PFF shape: torch.Size([2, 5, 64])


2 . Module d'attention multi-têtes

Objectif : Étendre le bloc à tête unique à une fonctionnalité à plusieurs têtes.

In [11]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout_rate=0.1):
        super(TransformerEncoderLayer, self).__init__()

        self.multi_head_attention = MultiHeadAttention(hidden_dim, num_heads)
        self.feed_forward = PositionwiseFeedForward(hidden_dim, ff_dim, dropout_rate)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        # Multi-Head Attention Block
        # Apply MHA, then dropout, then add residual connection, then LayerNorm
        attn_output, _ = self.multi_head_attention(x, x, x) # Self-attention
        x = self.norm1(x + self.dropout1(attn_output))

        # Position-wise Feed-Forward Block
        # Apply FFN, then dropout, then add residual connection, then LayerNorm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_output))

        return x

# --- Validation for TransformerEncoderLayer ---
print("\n--- Validating TransformerEncoderLayer Module ---")
batch_size = 2
seq_len = 5
hidden_dim = 64
num_heads = 8
ff_dim = 256
dropout_rate = 0.1

dummy_input_encoder = torch.randn(batch_size, seq_len, hidden_dim)
encoder_layer = TransformerEncoderLayer(hidden_dim, num_heads, ff_dim, dropout_rate)
output_encoder = encoder_layer(dummy_input_encoder)

print(f"Input TransformerEncoderLayer shape: {dummy_input_encoder.shape}")
print(f"Output TransformerEncoderLayer shape: {output_encoder.shape}") # Expected: (batch_size, seq_len, hidden_dim)


--- Validating TransformerEncoderLayer Module ---
Input TransformerEncoderLayer shape: torch.Size([2, 5, 64])
Output TransformerEncoderLayer shape: torch.Size([2, 5, 64])


3. (Optionnel) Pile d'encodeurs personnalisée et boucle d'entraînement

Objectif : Construire un réseau léger composé uniquement d'encodeurs.

In [12]:
# Install necessary libraries
!pip install datasets transformers accelerate -qq

In [13]:
import math
from datasets import load_dataset
from transformers import AutoTokenizer
import torch.optim as optim
from torch.utils.data import DataLoader

class CustomTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, max_seq_len, hidden_dim, num_heads, ff_dim, num_layers, num_classes, dropout_rate=0.1):
        super(CustomTransformerEncoder, self).__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.positional_embedding = nn.Embedding(max_seq_len, hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)

        self.layers = nn.ModuleList([
            TransformerEncoderLayer(hidden_dim, num_heads, ff_dim, dropout_rate)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, token_ids, attention_mask=None):
        seq_len = token_ids.shape[1]
        positions = torch.arange(0, seq_len, device=token_ids.device).unsqueeze(0)

        token_embeddings = self.token_embedding(token_ids)
        position_embeddings = self.positional_embedding(positions)

        x = self.dropout(token_embeddings + position_embeddings)

        for layer in self.layers:
            x = layer(x) # In this custom encoder, `x` is used for Q, K, V (self-attention)

        # For classification, typically take the representation of the first token (CLS token)
        # or average pooling over the sequence.
        # Here, we'll take the first token's representation.
        cls_output = x[:, 0, :]
        logits = self.classifier(cls_output)

        return logits

# --- Validation for CustomTransformerEncoder ---
print("\n--- Validating CustomTransformerEncoder Module ---")
vocab_size = 30000 # Example vocab size
max_seq_len = 128
hidden_dim = 64
num_heads = 8
ff_dim = 256
num_layers = 2
num_classes = 3 # For NLI (entailment, neutral, contradiction)
dropout_rate = 0.1

# Create dummy input token IDs and an optional attention mask
dummy_token_ids = torch.randint(0, vocab_size, (batch_size, max_seq_len))
dummy_attention_mask = torch.ones(batch_size, max_seq_len)

custom_encoder = CustomTransformerEncoder(vocab_size, max_seq_len, hidden_dim, num_heads, ff_dim, num_layers, num_classes, dropout_rate)
logits_custom = custom_encoder(dummy_token_ids)

print(f"Input token_ids shape: {dummy_token_ids.shape}")
print(f"Output logits shape: {logits_custom.shape}") # Expected: (batch_size, num_classes)


--- Validating CustomTransformerEncoder Module ---
Input token_ids shape: torch.Size([2, 128])
Output logits shape: torch.Size([2, 3])
